# CoordGen playground — two molecules

Edit the **Inputs** cell, then re-run **Render**.

| Field | Meaning |
| --- | --- |
| `REF_SMILES` / `QUERY_SMILES` | Molecules (RDKit parse only) |
| `MATCH` | Explicit `(query_idx, ref_idx)` pairs, or `None` for auto MCS / substructure |
| `CONSTRAINT` | `"constrained"` (soft), `"fixed"` (hard), or `"none"` (free query) |
| `FLAGS` | Native CoordGen layout knobs |

Layout is in-tree `xpict.layout2d` (Rust → CoordGen). Painting uses `depict_molecule`.

```bash
make build-python
uv run jupyter notebook notebooks/coordgen_playground.ipynb
```

In [ ]:
from __future__ import annotations

import json
import math

from IPython.display import HTML, display
from rdkit import Chem
from rdkit.Chem import rdFMCS
from rdkit.Chem.Draw import rdMolDraw2D

from xpict import _native
from xpict.contracts.scene import Scene
from xpict.draw.metrics import SCALE
from xpict.draw.svg import scene_to_svg
from xpict.layout2d import (
    LAYOUT_FLAG_BEST_PRECISION,
    LAYOUT_FLAG_EVEN_ANGLES,
    LAYOUT_FLAG_FORCE_OPEN_MACROCYCLES,
    LAYOUT_FLAG_METAL_BONDS_AS_ZOB,
    LAYOUT_FLAG_QUICK_PRECISION,
    LAYOUT_FLAG_SKIP_MINIMIZATION,
    Layout2DAtom,
    Layout2DBond,
    Layout2DRequest,
    Layout2DResult,
    generate_2d_coordinates,
)

assert _native.HAS_COORDGEN, "rebuild with make build-python"
COORDGEN_BOND = 50.0
TO_SVG = SCALE / COORDGEN_BOND
print("HAS_COORDGEN", _native.HAS_COORDGEN)

## Inputs

Change these and re-run the Render cell below. Atom indices are 0-based (see numbered maps).

In [ ]:
# --- edit me ---
REF_SMILES = "c1ccccc1O"
QUERY_SMILES = "COc1ccccc1"

# Explicit atom map query→ref, or None to auto-match (substructure / MCS).
# Example for phenol→anisole ring: MATCH = [(2,0),(3,1),(4,2),(5,3),(6,4),(7,5)]
MATCH: list[tuple[int, int]] | None = None

# How matched query atoms use ref coordinates:
CONSTRAINT = "constrained"  # "constrained" | "fixed" | "none"

# Native CoordGen flags (OR'd into layout_flags)
FLAGS = dict(
    even_angles=False,
    quick_precision=False,
    best_precision=False,
    force_open_macrocycles=False,
    metal_bonds_as_zob=False,
    skip_minimization=False,
)
# ---------------

FLAG_BITS = {
    "even_angles": LAYOUT_FLAG_EVEN_ANGLES,
    "quick_precision": LAYOUT_FLAG_QUICK_PRECISION,
    "best_precision": LAYOUT_FLAG_BEST_PRECISION,
    "force_open_macrocycles": LAYOUT_FLAG_FORCE_OPEN_MACROCYCLES,
    "metal_bonds_as_zob": LAYOUT_FLAG_METAL_BONDS_AS_ZOB,
    "skip_minimization": LAYOUT_FLAG_SKIP_MINIMIZATION,
}


def layout_flags_from(flags: dict[str, bool]) -> int:
    bits = 0
    for name, on in flags.items():
        if on:
            bits |= FLAG_BITS[name]
    return bits

In [ ]:
def mol_no_h(smiles: str) -> Chem.Mol:
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        raise ValueError(f"bad SMILES: {smiles!r}")
    try:
        Chem.Kekulize(m, clearAromaticFlags=True)
    except Chem.KekulizeException:
        pass
    return Chem.RemoveHs(m)


def request_from_mol(
    mol: Chem.Mol,
    *,
    constrained: dict[int, tuple[float, float]] | None = None,
    fixed: dict[int, tuple[float, float]] | None = None,
    layout_flags: int = 0,
) -> Layout2DRequest:
    atoms: list[Layout2DAtom] = []
    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        kw: dict = {
            "atomic_number": atom.GetAtomicNum(),
            "formal_charge": atom.GetFormalCharge(),
        }
        if fixed and i in fixed:
            x, y = fixed[i]
            kw.update(fixed=True, x=x, y=y)
        elif constrained and i in constrained:
            x, y = constrained[i]
            kw.update(constrained=True, x=x, y=y)
        atoms.append(Layout2DAtom(**kw))
    bonds = []
    for bond in mol.GetBonds():
        order = int(round(bond.GetBondTypeAsDouble())) or 1
        bonds.append(
            Layout2DBond(
                begin=bond.GetBeginAtomIdx(),
                end=bond.GetEndAtomIdx(),
                order=max(1, min(3, order)),
            )
        )
    return Layout2DRequest(atoms=atoms, bonds=bonds, layout_flags=layout_flags)


def resolve_match(
    ref: Chem.Mol, query: Chem.Mol, match: list[tuple[int, int]] | None
) -> list[tuple[int, int]]:
    if match is not None:
        return [(int(q), int(r)) for q, r in match]
    # GetSubstructMatch(pattern): result[pattern_idx] = mol_idx
    hit = query.GetSubstructMatch(ref)
    if hit:
        return [(int(qi), int(ri)) for ri, qi in enumerate(hit)]
    mcs = rdFMCS.FindMCS(
        [ref, query],
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        bondCompare=rdFMCS.BondCompare.CompareOrder,
        matchValences=True,
        ringMatchesRingOnly=True,
    )
    if mcs.numAtoms < 2:
        return []
    patt = Chem.MolFromSmarts(mcs.smartsString)
    if patt is None:
        return []
    rm = ref.GetSubstructMatch(patt)
    qm = query.GetSubstructMatch(patt)
    if not rm or not qm:
        return []
    return [(int(q), int(r)) for q, r in zip(qm, rm, strict=True)]


def paint_svg(mol: Chem.Mol, layout: Layout2DResult, *, title: str = "") -> str:
    atoms = []
    for i, atom in enumerate(mol.GetAtoms()):
        sym = atom.GetSymbol()
        atoms.append(
            {
                "index": i,
                "z": atom.GetAtomicNum(),
                "element": sym,
                "x": float(layout.x[i]) * TO_SVG,
                "y": float(layout.y[i]) * TO_SVG,
                "charge": int(atom.GetFormalCharge()),
                "label": None if sym == "C" else sym,
            }
        )
    bonds = [
        {
            "index": j,
            "begin": b.GetBeginAtomIdx(),
            "end": b.GetEndAtomIdx(),
            "order": float(b.GetBondTypeAsDouble() or 1.0),
        }
        for j, b in enumerate(mol.GetBonds())
    ]
    molecule = {"id": title or None, "atoms": atoms, "bonds": bonds}
    scene = Scene.model_validate(json.loads(_native.depict_molecule(json.dumps(molecule))))
    return scene_to_svg(scene)


def numbered_svg(mol: Chem.Mol, highlight: set[int] | None = None, w: int = 280, h: int = 220) -> str:
    """RDKit drawing with atom indices — for picking MATCH pairs."""
    m = Chem.Mol(mol)
    for a in m.GetAtoms():
        a.SetProp("atomNote", str(a.GetIdx()))
    Chem.rdDepictor.Compute2DCoords(m)
    d = rdMolDraw2D.MolDraw2DSVG(w, h)
    opts = d.drawOptions()
    opts.addAtomIndices = False
    opts.annotationFontScale = 0.7
    hl = list(highlight or [])
    rdMolDraw2D.PrepareAndDrawMolecule(d, m, highlightAtoms=hl)
    d.FinishDrawing()
    return d.GetDrawingText()


def show_row(title: str, panels: list[tuple[str, str]]):
    parts = [f"<h3>{title}</h3><div style='display:flex;gap:1rem;flex-wrap:wrap;align-items:flex-start'>"]
    for label, svg in panels:
        parts.append(
            f"<div><div style='font:12px sans-serif;margin-bottom:4px'>{label}</div>{svg}</div>"
        )
    parts.append("</div>")
    display(HTML("".join(parts)))

## Render

Re-run after editing Inputs. Numbered maps use RDKit 2D (for index picking only); the other panels are native CoordGen + xpict paint.

In [ ]:
ref_m = mol_no_h(REF_SMILES)
query_m = mol_no_h(QUERY_SMILES)
flags = layout_flags_from(FLAGS)
pairs = resolve_match(ref_m, query_m, MATCH)

ref_free = generate_2d_coordinates(request_from_mol(ref_m, layout_flags=flags))
query_free = generate_2d_coordinates(request_from_mol(query_m, layout_flags=flags))

seeds = {qi: (ref_free.x[ri], ref_free.y[ri]) for qi, ri in pairs}
if CONSTRAINT == "fixed" and seeds:
    query_aligned = generate_2d_coordinates(
        request_from_mol(query_m, fixed=seeds, layout_flags=flags)
    )
elif CONSTRAINT == "constrained" and seeds:
    query_aligned = generate_2d_coordinates(
        request_from_mol(query_m, constrained=seeds, layout_flags=flags)
    )
else:
    query_aligned = query_free

rms = None
if pairs:
    rms = math.sqrt(
        sum(
            (query_aligned.x[qi] - ref_free.x[ri]) ** 2
            + (query_aligned.y[qi] - ref_free.y[ri]) ** 2
            for qi, ri in pairs
        )
        / len(pairs)
    )

print(f"ref atoms={ref_m.GetNumAtoms()}  query atoms={query_m.GetNumAtoms()}")
print(f"match ({len(pairs)} atoms): {pairs}")
print(f"constraint={CONSTRAINT!r}  layout_flags={flags:#x}  FLAGS={FLAGS}")
if rms is not None:
    print(f"aligned RMS vs ref seeds: {rms:.3f} CoordGen units")

show_row(
    "Atom indices (edit MATCH from these)",
    [
        ("ref (numbered)", numbered_svg(ref_m, {r for _, r in pairs})),
        ("query (numbered)", numbered_svg(query_m, {q for q, _ in pairs})),
    ],
)

show_row(
    "Native CoordGen + xpict paint",
    [
        ("ref free", paint_svg(ref_m, ref_free, title="ref")),
        ("query free", paint_svg(query_m, query_free, title="q-free")),
        (f"query {CONSTRAINT}", paint_svg(query_m, query_aligned, title="q-align")),
    ],
)

## Preset examples

Uncomment one block, run it, then re-run **Render**.

In [ ]:
# Phenol → anisole (auto match)
REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = "c1ccccc1O", "COc1ccccc1", None, "constrained"

# Benzene → toluene
# REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = "c1ccccc1", "Cc1ccccc1", None, "constrained"

# Naphthalene → quinoline
# REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = "c1ccc2ccccc2c1", "c1ccc2ncccc2c1", None, "constrained"

# Manual ring map example (phenol O=6; anisole O=1, ring 2..7):
# REF_SMILES, QUERY_SMILES = "c1ccccc1O", "COc1ccccc1"
# MATCH = [(2, 0), (3, 1), (4, 2), (5, 3), (6, 4), (7, 5)]
# CONSTRAINT = "fixed"

# Flag experiments
# FLAGS["even_angles"] = True
# FLAGS["quick_precision"] = True
# FLAGS["best_precision"] = True
# FLAGS["skip_minimization"] = True

print("loaded preset — re-run Render cell")
print(REF_SMILES, "→", QUERY_SMILES, "CONSTRAINT", CONSTRAINT, "MATCH", MATCH)